# Saving inputs between sessions

Notebooks often start with a hard-coded block of inputs:

```python
LAYERS = [("Gypsum board", 12.7), ("Fiberglass insulation", 89), ("Brick", 90)]
```

A table is a friendlier way to hold them, but ordinarily its edits are lost when the kernel
restarts. Give a table a `persist` key and it saves its inputs whenever they change, and
restores them the next time a table with that key is created.

In [ ]:
from ipyrowtable.examples.conduction import LayerStack

wall = LayerStack(
    layers=[("Gypsum board", 12.7), ("Fiberglass insulation", 89), ("Brick", 90)],
    T_top=21,
    T_bottom=-10,
    persist="exterior wall",
)
wall

Edit the table, restart the kernel, and run this cell again: your edits come back.

* **Initial values.** The `layers=`, `T_top=` and `T_bottom=` arguments are now the
  *initial* values. They're used when nothing has been saved yet, when the saved inputs
  can't be used (a material that no longer exists, say), and by the **Reset** button.
* **What's saved.** Rows, boundary values, parameters, and which unit system was showing.
* **Where.** In `04_saving_inputs.ipyrowtable.json`, next to this notebook. If the notebook
  can't be identified, it goes in `ipyrowtable.json` in the working folder. The note under
  the table names the file; pass `persist_file=` to choose another. One file holds any number
  of tables, each under its own key.

In [ ]:
print(wall.persist_file)

## A new session, simulated

Re-running a cell after a restart builds a new table with the same key. Here the same thing
happens inside one session, using a separate key so your table above isn't touched.

In [ ]:
demo = LayerStack(persist="demo")
demo.reset()                           # start the demo from its initial values
demo.cell(1, "thickness").value = 140  # thicker insulation, in mm
demo.units = "Imperial"

again = LayerStack(persist="demo")     # what re-running the cell would give you
print(again.units.name, again.layers)

## What's in the file

Plain JSON, one entry per key, with values in *base* units (the units the compute function
uses: metres and °C here), so it doesn't matter which units were showing. It's a reasonable
thing to keep in version control next to the notebook.

In [ ]:
print(demo.persist_file.read_text())

## Several named sets of inputs

`get_inputs()` returns a JSON-ready snapshot and `set_inputs()` loads one, which is enough to
keep a few named designs and switch between them.

In [ ]:
again.units = "SI"
designs = {"as built": again.get_inputs()}
again.cell(1, "thickness").value = 200
designs["thicker insulation"] = again.get_inputs()

for name, inputs in designs.items():
    again.set_inputs(inputs)
    print(f"{name:20s} q = {again.results['q']:6.2f} W/m²")

## Starting over

* The **Reset** button returns to the initial values. It asks for a second click first.
  From code, use `table.reset()`.
* Deleting the file (or its entry) also starts every table from its initial values.
* If the saved inputs can't be used, the table starts from its initial values and says why.
  The old file is kept as a `.bak` copy before it is next written.

In [ ]:
again.reset()
print(again.layers)

## Turning it off

Leave out `persist` (the default) and a table always starts from its initial values.

To turn saving off for **every** table, set the environment variable `IPYROWTABLE_PERSIST=off`
before creating them. This is useful for batch runs that must start from known inputs.

In [ ]:
import os

os.environ["IPYROWTABLE_PERSIST"] = "off"
fixed = LayerStack(persist="exterior wall")
print(fixed.persist_file, fixed.layers)  # None: this table neither restores nor saves
del os.environ["IPYROWTABLE_PERSIST"]